# CEVAE実験: 潜在交絡因子Zが2次元の場合 (元 CEVAE実験_v5.ipynb)

1次元実験(`run_1d_experiment.ipynb`)のリベンジとして、真のZを2次元に戻してCEVAE vs X-learnerを比較する。
1次元実験で見つかった「CEVAEのrunが成功/崩壊の二山に割れる」問題に対し、崩壊を検出する信号
`mse_ratio`(観測データだけで計算可能)で崩壊runを弾く「崩壊ガード」を使う。

結果は `results/2d_main_results.csv` に保存する(1次元との比較は `compare_1d_vs_2d.ipynb` で行う)。

計算量の注意: 元の設定(ハイパラ探索n_iter=60、10 data_seed×5 model_seed restart)は重い。
まず動作を見るだけなら `N_ITER_CEVAE` や `N_RESTARTS` を小さくするとよい。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul.*", category=RuntimeWarning)

import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from dgp import generate_toy_2d
# xgboost(X-learner側)を先に読み込む: 先にtorchを読み込むと、環境によっては
# XGBoostの学習時にクラッシュすることがあるため(OpenMPライブラリの競合)。
from xlearner_baseline import run_xlearner, refit_xlearner_with_fixed_params, predict_xlearner

import torch
from cevae_model import CEVAEConfig, build_cevae, train_cevae, compute_val_elbo, estimate_ite
from diagnostics import (
    z_recovery_score, compute_y_mse, guard_threshold,
    confusion_at_threshold, best_threshold,
)

## データ生成(data_seed=0)とX-learnerベースライン

In [ ]:
DATA_SEED = 0
X, T, Y, Y0, Y1, tau, Z = generate_toy_2d(seed=DATA_SEED)
d = X.shape[1]
N = len(Y)
proxy_cols = [f"X{j+1}" for j in range(d)]

df_full = pd.DataFrame(X, columns=proxy_cols)
df_full["T"] = T
df_full["Y"] = Y
df_full["tau"] = tau
df_train, df_test = train_test_split(df_full, test_size=0.3, random_state=DATA_SEED, stratify=df_full["T"])

print(f"N={N}, d={d}, train={len(df_train)}, test={len(df_test)}")
print(f"真のATE: {tau.mean():.3f}   ナイーブATE: {Y[T==1].mean()-Y[T==0].mean():.3f}")

N_ITER_XGB = 30  # 元の設定
CV_XGB = 3

print("\nX-learner ハイパラ探索中...")
xl_tau_pred, xl_models_fixed = run_xlearner(df_train, df_test, proxy_cols, n_iter=N_ITER_XGB, cv=CV_XGB, seed=0)
xl_pehe_s0 = np.sqrt(np.mean((df_test["tau"].values - xl_tau_pred) ** 2))
print(f"X-learner √PEHE (data_seed=0): {xl_pehe_s0:.3f}")

## CEVAEのハイパラ探索(z_dimも含めて探索 + 崩壊ガード)

`mse_ratio`(zを通すとYの予測が何倍悪化するか)が異常に大きいrunを「崩壊」として弾いてから、
残ったrunの中でval_ELBOが一番良いものを採用する。

In [ ]:
def df_to_tensors(df):
    return (
        torch.tensor(df[proxy_cols].values, dtype=torch.float32),
        torch.tensor(df["T"].values, dtype=torch.float32),
        torch.tensor(df["Y"].values, dtype=torch.float32),
    )

df_inner, df_val = train_test_split(df_train, test_size=0.2, random_state=0, stratify=df_train["T"])
x_inner, t_inner, y_inner = df_to_tensors(df_inner)
x_val, t_val, y_val = df_to_tensors(df_val)
y_val_np = df_val["Y"].values

SEARCH_SPACE = {
    "z_dim": [1, 2, 3],
    "hidden": [32, 64, 128],
    "beta_kl_max": [0.1, 0.3, 0.5, 1.0],
    "kl_anneal_epochs": [50, 100, 150],
    "epochs": [150, 300],
    "weight_decay": [0.0, 1e-4, 1e-3],
    "lr": [1e-3, 5e-4, 3e-4],
}
N_ITER_CEVAE = 60  # 元の設定

rng_search = random.Random(42)
search_records = []
cfg_list = []  # search_recordsと同じ順番でcfgそのものも保持しておく(DataFrame経由だと型が壊れるため)

print(f"CEVAE ランダムサーチ中(z_dimも含めて探索, n_iter={N_ITER_CEVAE})...")
for i in range(N_ITER_CEVAE):
    cfg = CEVAEConfig(
        z_dim=rng_search.choice(SEARCH_SPACE["z_dim"]),
        hidden=rng_search.choice(SEARCH_SPACE["hidden"]),
        beta_kl_max=rng_search.choice(SEARCH_SPACE["beta_kl_max"]),
        kl_anneal_epochs=rng_search.choice(SEARCH_SPACE["kl_anneal_epochs"]),
        epochs=rng_search.choice(SEARCH_SPACE["epochs"]),
        weight_decay=rng_search.choice(SEARCH_SPACE["weight_decay"]),
        lr=rng_search.choice(SEARCH_SPACE["lr"]),
        seed=0,
    )
    nets = build_cevae(cfg, d)
    train_cevae(nets, cfg, x_inner, t_inner, y_inner)
    val_elbo = compute_val_elbo(nets, x_val, t_val, y_val)
    _, _, mse_ratio = compute_y_mse(nets, x_val, t_val, y_val_np)
    search_records.append({"iter": i, "val_elbo": val_elbo, "mse_ratio": mse_ratio, **cfg.__dict__})
    cfg_list.append(cfg)

df_search = pd.DataFrame(search_records)

guard_th = guard_threshold(df_search["mse_ratio"])
df_search["guard_pass"] = df_search["mse_ratio"] <= guard_th
n_excluded = int((~df_search["guard_pass"]).sum())
best_row = df_search[df_search["guard_pass"]].sort_values("val_elbo").iloc[0]
best_cfg = cfg_list[int(best_row["iter"])]

print(f"\n崩壊ガード閾値(mse_ratio) = {guard_th:.3f} → {n_excluded}/{N_ITER_CEVAE} trialを除外")
print(f"採用設定: z_dim={best_cfg.z_dim}  val_elbo={best_row['val_elbo']:.3f}  mse_ratio={best_row['mse_ratio']:.3f}")

BEST_CEVAE_CFG = CEVAEConfig(**{**best_cfg.__dict__, "seed": 0})
print(f"\n以降のマルチシード実験で使う採用設定: {BEST_CEVAE_CFG}")

## メイン実験: 10 data_seed × 5 model_seed(restart)でCEVAE vs X-learnerを比較

In [ ]:
N_DATA_SEEDS = 10
N_RESTARTS = 5  # data_seedごとのmodel_seed再挑戦回数(初期値運があるため複数回試す)

multi_records = []

for ds in range(N_DATA_SEEDS):
    print(f"\n=== data_seed={ds} ===", flush=True)
    Xs, Ts, Ys, _, _, taus, Zs = generate_toy_2d(seed=ds)
    df_s = pd.DataFrame(Xs, columns=proxy_cols)
    df_s["T"] = Ts
    df_s["Y"] = Ys
    df_s["tau"] = taus
    df_tr_s, df_te_s = train_test_split(df_s, test_size=0.3, random_state=ds, stratify=df_s["T"])
    tau_true_s = df_te_s["tau"].values
    te_idx = df_te_s.index.tolist()
    Z_te_s = Zs[te_idx]

    df_inner_s, df_val_s = train_test_split(df_tr_s, test_size=0.2, random_state=ds, stratify=df_tr_s["T"])
    x_inner_s, t_inner_s, y_inner_s = df_to_tensors(df_inner_s)
    x_val_s, t_val_s, y_val_s = df_to_tensors(df_val_s)
    y_val_s_np = df_val_s["Y"].values

    xt = torch.tensor(Xs, dtype=torch.float32)
    tt = torch.tensor(Ts, dtype=torch.float32)
    yt = torch.tensor(Ys, dtype=torch.float32)
    x_te_s, t_te_s, y_te_s = xt[te_idx], tt[te_idx], yt[te_idx]

    # X-learner: data_seed=0の固定パラメータで再学習
    models_s = refit_xlearner_with_fixed_params(xl_models_fixed, df_tr_s, proxy_cols)
    tau_xl = predict_xlearner(models_s, df_te_s, proxy_cols)
    xl_pehe_s = float(np.sqrt(np.mean((tau_true_s - tau_xl) ** 2)))
    xl_eps_ate_s = float(abs(tau_xl.mean() - tau_true_s.mean()))

    # CEVAE: 採用設定(BEST_CEVAE_CFG)を固定し、model_seedだけN_RESTARTS回振る
    for ms in range(N_RESTARTS):
        cfg_ms = CEVAEConfig(**{**BEST_CEVAE_CFG.__dict__, "seed": ms})
        nets_ms = build_cevae(cfg_ms, d)
        train_cevae(nets_ms, cfg_ms, x_inner_s, t_inner_s, y_inner_s)

        # 選択信号(真のtau・Zを使わない。held-outのvalで計算)
        val_elbo_ms = compute_val_elbo(nets_ms, x_val_s, t_val_s, y_val_s)
        mse_q_ms, mse_p_ms, mse_ratio_ms = compute_y_mse(nets_ms, x_val_s, t_val_s, y_val_s_np)

        # 答え合わせ専用(真のtau・Zを使う。testで計算)
        tau_pred_ms = estimate_ite(nets_ms, x_te_s)
        pehe_ms = float(np.sqrt(np.mean((tau_true_s - tau_pred_ms) ** 2)))
        eps_ate_ms = float(abs(tau_pred_ms.mean() - tau_true_s.mean()))
        zrec_ms = z_recovery_score(nets_ms, x_te_s, t_te_s, y_te_s, Z_te_s)

        multi_records.append({
            "data_seed": ds, "model_seed": ms, "z_dim": cfg_ms.z_dim,
            "val_elbo": val_elbo_ms, "mse_ratio": mse_ratio_ms,
            "sqrt_pehe": pehe_ms, "eps_ate": eps_ate_ms, "z_recovery": zrec_ms,
            "xl_pehe": xl_pehe_s, "xl_eps_ate": xl_eps_ate_s,
        })
        print(f"  ms={ms}  PEHE={pehe_ms:.3f}  z_recovery={zrec_ms:.3f}  elbo={val_elbo_ms:.3f}  mse_ratio={mse_ratio_ms:.3f}")

df_multi = pd.DataFrame(multi_records)
df_multi.to_csv("results/2d_main_results.csv", index=False)
print("\n→ results/2d_main_results.csv に保存しました")

## 崩壊ガードを適用して集計

In [ ]:
GUARD_TH_MULTI = guard_threshold(df_multi["mse_ratio"])
df_multi["guard_pass"] = df_multi["mse_ratio"] <= GUARD_TH_MULTI
df_pass = df_multi[df_multi["guard_pass"]]

print(f"崩壊ガード閾値 = {GUARD_TH_MULTI:.3f}")
print(f"ガード通過: {int(df_multi['guard_pass'].sum())}/{len(df_multi)} run")

print("\n=== 崩壊ガード適用後のCEVAE成績(生存runのみ、mean±std) ===")
print(df_pass[["sqrt_pehe", "eps_ate", "z_recovery"]].agg(["mean", "std"]).round(3))

xl_summary_df = df_multi.drop_duplicates("data_seed")[["data_seed", "xl_pehe", "xl_eps_ate"]].set_index("data_seed")
print("\n=== X-learner成績(data_seedごとに1つ、mean±std) ===")
print(xl_summary_df.agg(["mean", "std"]).round(3))

per_seed_cevae = df_pass.groupby("data_seed")["sqrt_pehe"].mean()
common_seeds = per_seed_cevae.index.intersection(xl_summary_df.index)
n_win = int((per_seed_cevae.loc[common_seeds] < xl_summary_df["xl_pehe"].loc[common_seeds]).sum())
print(f"\nCEVAE勝率(seed別、ガード後平均で比較): {n_win}/{len(common_seeds)}")

## mse_ratio vs val_ELBOで、崩壊検出の分離能力を比較

In [ ]:
Z_RECOVERY_SUCCESS_TH = 0.5
df_multi["collapsed_truth"] = df_multi["z_recovery"] < Z_RECOVERY_SUCCESS_TH

fig, ax = plt.subplots(1, 1, figsize=(7, 5))
for collapsed, color, label in [(False, "#2ecc71", "成功"), (True, "#e74c3c", "崩壊")]:
    sub = df_multi[df_multi["collapsed_truth"] == collapsed]
    ax.scatter(sub["mse_ratio"], sub["sqrt_pehe"], c=color, label=label, alpha=0.85, edgecolor="black", linewidth=0.4, s=60)
ax.axvline(GUARD_TH_MULTI, color="gray", linestyle="--", label=f"ガード閾値={GUARD_TH_MULTI:.2f}")
ax.set_xlabel("mse_ratio = mse_p / mse_q")
ax.set_ylabel("√PEHE")
ax.set_title("2次元Zでもmse_ratioで二山が分離するか")
ax.legend()
plt.tight_layout()
plt.show()

comparison = pd.DataFrame([best_threshold(df_multi, c, true_col="collapsed_truth") for c in ["val_elbo", "mse_ratio"]])
comparison